# 07 - Connectivity groups vs. kinematic trajectories

**Purpose**: close the analytical loop. Notebooks 01-06 identified
which connectivity and kinematic features carry most of the variance
and which kinematic features change across phases. This notebook
asks the payoff question: **do mice with similar connectivity
profiles show similar kinematic trajectories across phases?**

Pipeline implemented here:

1. Cluster subjects on connectivity (ward / kmeans / gmm / consensus).
2. Profile each cluster: which regions define it, relative to the
   overall population? Auto-generate human-readable labels.
3. Validate the clustering with LOO + random-permutation nulls.
4. Alluvial / Sankey: do subjects clustered on baseline kinematics
   stay together across phases, or mix?
5. Continuous trajectories colored by connectivity PC1.
6. Grouped trajectories colored by named clusters.
7. Interaction LMM: ``feature ~ phase * cluster`` with nested
   subject/session random effects.
8. Ascending-connectivity placeholder: where the second-pass
   analysis plugs in when ascending tracing data arrives.

**Synthetic-cohort mode**. When the real cohort is small the
clustering or interaction statistics can be uninformative on
their own. Set ``USE_SYNTHETIC = True`` in the parameters cell to
swap in a cloned-and-perturbed synthetic cohort so the pipeline
can be validated at realistic N. Synthetic runs produce output
that exercises every code path but is explicitly NOT real data;
every figure generated in this mode should be labeled as such
in any writeup.

See [`../docs/assumptions.md`](../docs/assumptions.md) for the
N-scaling story and why the per-phase kinematic clustering used
for the alluvial plot doesn't double-dip against connectivity.


In [ ]:
# parameters
#
# --- Data source ---
USE_SYNTHETIC = False                # True = pipeline validation mode; False = real data
SYNTHETIC_N = 30                     # how many synthetic subjects to mint
SYNTHETIC_SEED = 42                  # RNG seed for reproducibility
SYNTHETIC_CONN_NOISE = 0.3           # cross-subject-std fraction added to connectivity clones
SYNTHETIC_KINE_NOISE = 0.10          # per-feature-std fraction added to reach-level clones
#
# --- Clustering ---
CLUSTER_METHOD = 'ward'              # 'ward', 'kmeans', 'gmm', 'consensus'
N_CLUSTERS = 4                       # Bump up when N grows
N_CONN_PCS = 3                       # Connectivity PCs used for the coordinate axis
#
# --- Cluster profiling ---
PROFILE_TOP_N = 2                    # regions mentioned per auto-generated cluster name
PROFILE_THRESHOLD = 0.5              # |z-score| below this is treated as "not distinctive"
MANUAL_CLUSTER_NAMES = {}            # Optional user override: {int_cluster_id: 'biological name'}
#
# --- Permutation validation ---
N_PERMUTATIONS = 500                 # random-label shuffles for the null
#
# --- Trajectory display ---
TRAJECTORY_FEATURE = 'max_extent_mm'
AGG_STAT = 'mean'                    # '_mean', '_std', '_median', '_q25', '_q75'
PHASES = ['Baseline', 'Post_Injury_1', 'Post_Injury_2-4', 'Post_Rehab_Test']
#
# --- LMM ---
RUN_INTERACTION_LMM = True
#
# --- Figure sizes ---
FIGSIZE_TRAJ = (12, 6)
FIGSIZE_PROFILE = (14, 6)
FIGSIZE_PERMUTATION = (10, 5)
FIGSIZE_ALLUVIAL = (900, 500)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from endpoint_ck_analysis import SKILLED_REACHING, ordered_hemisphere_columns
from endpoint_ck_analysis.config import CACHE_DIR, EXAMPLE_OUTPUT_DIR, ANALYZABLE_PHASES
from endpoint_ck_analysis.data_loader import load_all
from endpoint_ck_analysis.helpers.clusters import (
    cluster_subjects, profile_clusters, auto_name_clusters,
    permutation_validate, alluvial_source_records,
)
from endpoint_ck_analysis.helpers.plotting import stamp_version

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
data = load_all(
    use_synthetic=USE_SYNTHETIC,
    synthetic_n=SYNTHETIC_N,
    synthetic_seed=SYNTHETIC_SEED,
    synthetic_conn_noise=SYNTHETIC_CONN_NOISE,
    synthetic_kine_noise=SYNTHETIC_KINE_NOISE,
    use_cache=not USE_SYNTHETIC,     # synthetic always rebuilds
    write_cache=not USE_SYNTHETIC,
    verbose=False,
)
print(f"Running on {'SYNTHETIC' if USE_SYNTHETIC else 'REAL'} data.  "
      f"N={len(data.matched_subjects)} subjects.")


## 1. Compute connectivity coordinates

Re-fit PCA on the full connectivity matrix (same matrix as notebook
01). Each subject ends up with PC1/PC2/PC3 scores that serve as a
continuous coordinate for coloring trajectories.


In [ ]:
X_conn = data.FCDGdf_wide.fillna(0)
canonical_cols = ordered_hemisphere_columns(SKILLED_REACHING, available=X_conn.columns.tolist())
X_conn = X_conn[canonical_cols]

X_scaled = StandardScaler().fit_transform(X_conn)
conn_pca = PCA(n_components=min(N_CONN_PCS, len(X_conn) - 1))
conn_scores = conn_pca.fit_transform(X_scaled)
conn_scores_df = pd.DataFrame(
    conn_scores,
    index=X_conn.index,
    columns=[f'PC{i+1}' for i in range(conn_scores.shape[1])],
)
conn_scores_df.index.name = 'subject_id'
print('Per-subject connectivity PC scores:')
print(conn_scores_df)

# Cache these so downstream notebooks can also colour by them
conn_scores_df.to_parquet(CACHE_DIR / 'connectivity_pc_scores.parquet')


## 2. Cluster subjects on connectivity

Pick a clustering method via ``CLUSTER_METHOD`` in the parameters
cell. All four options go through the same ``cluster_subjects``
helper and return a 1..K label per subject.

- ``ward`` -- deterministic hierarchical clustering.
- ``kmeans`` -- fast, needs a seed for reproducibility.
- ``gmm`` -- Gaussian mixture; produces soft probabilities too.
- ``consensus`` -- bootstrap-resampled k-means with modal vote,
  more stable at higher N.

At N_CLUSTERS equal to the matched-subject count each subject gets its own cluster;
honest outcome. Under synthetic mode with the default noise scale,
every method recovers the ground-truth prototype assignment at
100% (verified during development).


In [ ]:
cluster_result = cluster_subjects(
    X_conn, method=CLUSTER_METHOD, k=N_CLUSTERS, random_state=SYNTHETIC_SEED,
)
cluster_by_subject = cluster_result.labels.rename('conn_cluster').astype(int)
print(f'Method: {cluster_result.method}, k={cluster_result.k}')
print('\nCluster assignments:')
print(cluster_by_subject)


## 3. Cluster profiling + naming

For each cluster, compute the z-score of each connectivity region
relative to the overall population. Big absolute z => that region
defines that cluster. Auto-generate short labels like
``"cluster3: up-Red_Nucleus_both down-Corticospinal_left"``.
Override with biological names in ``MANUAL_CLUSTER_NAMES`` once
they're known.


In [ ]:
profile = profile_clusters(X_conn, cluster_by_subject)
auto_names = auto_name_clusters(profile, top_n=PROFILE_TOP_N, threshold=PROFILE_THRESHOLD)
names = dict(auto_names)
names.update(MANUAL_CLUSTER_NAMES)
print('Cluster names:')
for cid in sorted(names):
    print(f'  {names[cid]}')

# Heatmap: clusters (rows) x top defining regions (cols)
top_cols = (profile.abs().max(axis=0).sort_values(ascending=False).head(20).index.tolist())
fig, ax = plt.subplots(figsize=FIGSIZE_PROFILE)
sns.heatmap(profile[top_cols].rename(index=names), cmap='RdBu_r', center=0,
            annot=True, fmt='.1f', ax=ax, cbar_kws={'label': 'z-score'})
ax.set_title('Cluster profile: top 20 discriminating regions')
plt.tight_layout()
stamp_version(fig, label='07 cluster profile')
plt.savefig(EXAMPLE_OUTPUT_DIR / '07_cluster_profile.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Permutation validation

Two null distributions for the "are clusters real?" question:

- **LOO**: recompute within-cluster variance with one subject
  removed at a time. Narrow spread => clustering is robust to
  any single subject.
- **Random-equal-N**: shuffle the cluster labels ``N_PERMUTATIONS``
  times, preserving group sizes; recompute within-cluster
  variance. If the observed statistic falls in the left tail of
  this null, clusters are tighter than chance.

``p_random`` is the fraction of shuffled draws with
within-cluster variance <= observed. A small p (<0.05) says the
clustering captures real structure.


In [ ]:
pv = permutation_validate(X_conn, cluster_by_subject,
                          n_random=N_PERMUTATIONS, random_state=SYNTHETIC_SEED)
print(f'Observed within-cluster variance: {pv["observed"]:.2f}')
print(f'LOO range:                        [{min(pv["loo"]):.2f}, {max(pv["loo"]):.2f}]')
print(f'Random null mean:                 {float(np.mean(pv["random"])):.2f}')
print(f'p (random <= observed):           {pv["p_random"]:.3f}')

fig, ax = plt.subplots(figsize=FIGSIZE_PERMUTATION)
ax.hist(pv['random'], bins=50, alpha=0.7, label='Random-label null', color='grey')
ax.axvline(pv['observed'], color='red', linewidth=2, label=f'Observed ({pv["observed"]:.1f})')
for v in pv['loo']:
    ax.axvline(v, color='blue', alpha=0.3, linewidth=0.8)
ax.set_xlabel('Within-cluster variance (sum across features)')
ax.set_ylabel('Count of null draws')
ax.set_title(f'Cluster validity (p={pv["p_random"]:.3f}; blue ticks = LOO)')
ax.legend()
plt.tight_layout()
stamp_version(fig, label='07 permutation')
plt.savefig(EXAMPLE_OUTPUT_DIR / '07_permutation_validation.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Alluvial: do subjects cluster the same way per phase?

Cluster subjects separately within each phase based on their
kinematic profile at that phase. An alluvial / Sankey flow shows
which subjects stay in the same kinematic cluster across phases
and which migrate. Heavy flow between aligned clusters means
subjects have consistent kinematic phenotypes through time;
crossing flows mean kinematic clusters shuffle.

Double-dipping note: this clustering uses kinematic data only,
not connectivity. It is an independent structural question from
the connectivity clustering above.


In [ ]:
agg_flat = data.AKDdf_agg_contact_flat()
kine_feature_cols = [c for c in agg_flat.columns if c.endswith(f'_{AGG_STAT}')]

per_phase_labels = {}
for phase in PHASES:
    phase_slice = agg_flat[
        (agg_flat['phase_group'] == phase) & (agg_flat['contact_group'] == 'contacted')
    ]
    if phase_slice.empty:
        continue
    mat = (
        phase_slice.set_index('subject_id')[kine_feature_cols]
        .fillna(0)
    )
    try:
        k_phase = min(N_CLUSTERS, len(mat) - 1) if len(mat) > 1 else 1
        cr = cluster_subjects(mat, method=CLUSTER_METHOD, k=max(k_phase, 2), random_state=SYNTHETIC_SEED)
        per_phase_labels[phase] = cr.labels
    except Exception as e:
        print(f'Phase {phase}: clustering failed ({e})')

sankey_df = alluvial_source_records(per_phase_labels, PHASES)
print(f'Sankey edges: {len(sankey_df)} between {len(per_phase_labels)} phases.')


In [ ]:
import plotly.graph_objects as go

if len(sankey_df) == 0:
    print('Not enough per-phase clustering data to draw the Sankey.')
else:
    nodes = pd.unique(pd.concat([sankey_df['source'], sankey_df['target']]))
    node_index = {n: i for i, n in enumerate(nodes)}
    fig_sankey = go.Figure(data=[go.Sankey(
        node=dict(label=list(nodes), pad=12, thickness=14),
        link=dict(
            source=sankey_df['source'].map(node_index).tolist(),
            target=sankey_df['target'].map(node_index).tolist(),
            value=sankey_df['value'].tolist(),
        ),
    )])
    fig_sankey.update_layout(
        title_text='Kinematic-cluster flow across phases',
        width=FIGSIZE_ALLUVIAL[0], height=FIGSIZE_ALLUVIAL[1],
    )
    try:
        fig_sankey.write_image(str(EXAMPLE_OUTPUT_DIR / '07_alluvial.png'))
    except Exception as e:
        print(f'(Could not save Sankey PNG -- kaleido missing or failed: {e})')
    fig_sankey.show()


## 6. Build the per-subject per-phase trajectory table

Pull ``{FEATURE}_{AGG_STAT}`` from ``data.AKDdf_agg_contact`` for the
contacted reaches, restricted to the analyzable phase set. One row
per subject per phase.


In [ ]:
agg_flat = data.AKDdf_agg_contact_flat()
feature_col = f'{TRAJECTORY_FEATURE}_{AGG_STAT}'
if feature_col not in agg_flat.columns:
    raise KeyError(
        f'{feature_col!r} not in AKDdf_agg_contact columns. '
        f'Set TRAJECTORY_FEATURE / AGG_STAT to a valid combination. '
        f'First few available: {[c for c in agg_flat.columns if c.endswith("_" + AGG_STAT)][:10]}'
    )

traj = agg_flat[
    (agg_flat['contact_group'] == 'contacted')
    & (agg_flat['phase_group'].isin(PHASES))
][['subject_id', 'phase_group', feature_col]].copy()
traj['phase_order'] = traj['phase_group'].apply(lambda p: PHASES.index(p) if p in PHASES else -1)

# Attach connectivity PC1 score and cluster to each row, drop subjects without connectivity
traj = traj.merge(conn_scores_df[['PC1']].reset_index(), on='subject_id', how='left')
traj = traj.merge(cluster_by_subject.to_frame().reset_index(), on='subject_id', how='left')
traj = traj.dropna(subset=['conn_cluster']).copy()
traj['conn_cluster'] = traj['conn_cluster'].astype(int)
print(traj.sort_values(['subject_id', 'phase_order']).head(20))


## 7. Continuous view: trajectories colored by connectivity PC1

One line per subject, x = phase (ordered), y = chosen feature.
Line color is a continuous mapping of that subject's connectivity
PC1 score. No grouping imposed. At small N this is the most honest view
-- it shows whether the connectivity gradient corresponds to any
visible trajectory structure without pretending there are discrete
groups.


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE_TRAJ)
pc1_vals = conn_scores_df['PC1']
vmin, vmax = pc1_vals.min(), pc1_vals.max()
cmap = get_cmap('viridis')
for subj, grp in traj.groupby('subject_id'):
    grp = grp.sort_values('phase_order')
    color = cmap((grp['PC1'].iloc[0] - vmin) / (vmax - vmin + 1e-9))
    ax.plot(grp['phase_order'], grp[feature_col], '-o', color=color, label=subj, linewidth=2)
    ax.annotate(subj, (grp['phase_order'].iloc[-1], grp[feature_col].iloc[-1]),
                fontsize=8, xytext=(4, 0), textcoords='offset points')
ax.set_xticks(range(len(PHASES)))
ax.set_xticklabels(PHASES, rotation=20, ha='right')
ax.set_ylabel(feature_col)
ax.set_xlabel('Phase')
ax.set_title(f'Trajectory of {feature_col} colored by connectivity PC1')
sm = plt.cm.ScalarMappable(cmap=cmap,
                           norm=plt.Normalize(vmin=vmin, vmax=vmax))
plt.colorbar(sm, ax=ax, label='Connectivity PC1 score')
plt.tight_layout()
stamp_version(fig, label='07 continuous')
plt.savefig(EXAMPLE_OUTPUT_DIR / '07_trajectories_continuous.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Grouped view: trajectories by named connectivity cluster

Same trajectories, colored by cluster membership and labeled with
the auto-generated or manual names from section 3.


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE_TRAJ)
palette = get_cmap('tab10')
already_labeled = set()
for cluster_id in sorted(traj['conn_cluster'].unique()):
    grp = traj[traj['conn_cluster'] == cluster_id].sort_values(['subject_id', 'phase_order'])
    color = palette((int(cluster_id) - 1) % 10)
    cluster_name = names.get(int(cluster_id), f'cluster{cluster_id}')
    for subj, sub in grp.groupby('subject_id'):
        label = cluster_name if cluster_name not in already_labeled else None
        ax.plot(sub['phase_order'], sub[feature_col], '-o', color=color, alpha=0.85,
                label=label, linewidth=2)
        already_labeled.add(cluster_name)
ax.set_xticks(range(len(PHASES)))
ax.set_xticklabels(PHASES, rotation=20, ha='right')
ax.set_ylabel(feature_col)
ax.set_xlabel('Phase')
ax.set_title(f'Trajectory of {feature_col} by connectivity cluster (k={N_CLUSTERS})')
ax.legend(loc='best', fontsize=7)
plt.tight_layout()
stamp_version(fig, label='07 grouped')
plt.savefig(EXAMPLE_OUTPUT_DIR / '07_trajectories_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Interaction LMM template

Asks whether the phase effect on the feature differs by
connectivity cluster, using the same nested random-effects
structure as notebook 05.

Formula: ``feature ~ C(phase_group) * C(conn_cluster)``
with random intercept for subject_id and nested session-within-
subject.

When the cluster count equals the subject count this is statistically vacuous -- the
interaction is a reparameterization rather than a tested effect.
Set ``RUN_INTERACTION_LMM = False`` to skip. When N grows and
clusters contain multiple subjects this becomes the primary
inferential test that answers "do connectivity groups follow
different kinematic trajectories?".


In [ ]:
import warnings
if RUN_INTERACTION_LMM and traj['conn_cluster'].nunique() > 1:
    from statsmodels.formula.api import mixedlm

    reach_level = data.AKDdf[data.AKDdf['contact_group'] == 'contacted'].copy()
    reach_level['phase_group'] = pd.Categorical(
        reach_level['phase_group'], categories=PHASES, ordered=True
    )
    # Attach connectivity cluster ID by subject_id
    reach_level = reach_level.merge(cluster_by_subject.to_frame().reset_index(),
                                    on='subject_id', how='left')
    # Drop subjects without a cluster assignment (no connectivity data)
    subset = reach_level.dropna(subset=['conn_cluster', TRAJECTORY_FEATURE])
    subset['conn_cluster'] = subset['conn_cluster'].astype(int).astype(str)
    if subset['subject_id'].nunique() >= 2 and subset['conn_cluster'].nunique() >= 2:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                model = mixedlm(
                    formula=f"Q('{TRAJECTORY_FEATURE}') ~ C(phase_group) * C(conn_cluster)",
                    data=subset,
                    groups='subject_id',
                    vc_formula={'session': '0 + C(session_date)'},
                )
                result = model.fit(reml=True, method='lbfgs', disp=False)
            wald = result.wald_test_terms().table
            print(wald)
            print('\nInteraction p-value for phase x conn_cluster:')
            interaction_rows = [i for i in wald.index if 'phase_group' in i and 'conn_cluster' in i]
            for i in interaction_rows:
                print(f'  {i}: {wald.loc[i, "P>chi2"]:.4f}')
        except Exception as e:
            print(f'Interaction LMM failed to fit (expected at very small N): {e}')
    else:
        print('Too few subjects or clusters to fit the interaction LMM.')
else:
    print('Skipping interaction LMM (RUN_INTERACTION_LMM=False or only one cluster present).')


## 10. Ascending connectivity placeholder

When ascending-projection tracing data lands in mousedb (new
table: ``ascending_region_counts`` or equivalent), the analysis
plugs in here:

**Question**: within each descending-connectivity cluster (i.e.
for mice with similar motor-output sparing profiles), does
variability in ascending connectivity predict kinematic
trajectory differences?

**Implementation sketch**:

1. Load ascending counts alongside descending: add a
   ``data.ascendingdf`` section to ``data_loader``.
2. For each ``conn_cluster`` in this notebook, subset subjects
   and compute an ascending-connectivity PC score.
3. Regress the kinematic trajectory (e.g. per-phase PC1 scores)
   onto the ascending score with subject as random effect.
4. Partial-out the descending contribution so residual kinematic
   differences are attributed only to ascending variability.

At current N and without ascending data this remains a note.
When the cohort and data are ready, the pattern above slots in
without changing the earlier notebooks.


<!-- INTERPRETATION_BLOCK -->
## How to read this notebook's output

<details>
<summary>What the trajectory + cluster outputs tell you (click to expand)</summary>

**Cluster profile heatmap (z-score per cluster per region)**: which
connectivity regions define each cluster.

- Clusters with strongly red OR strongly blue cells in distinct
  regions = clusters genuinely differ in their connectivity profile;
  the auto-generated names ("up-Corticospinal_both", etc.) reflect
  real defining features.
- Clusters with mostly pale colors = subjects in those clusters are
  near the population mean in most regions; clusters are "central"
  and don't have strong defining features.

**Permutation validation panel** (histogram + vertical lines):

- Observed within-cluster variance (red) far below the null
  distribution mean (gray histogram) = clustering captures real
  structure; subjects within a cluster genuinely look more alike than
  random groups of the same size.
- Observed near or above the null distribution = clusters are noise;
  the algorithm divided subjects but no within-cluster coherence
  beyond chance.
- LOO ticks (blue) clustered tightly around observed = result is
  robust to dropping any single subject. LOO ticks spread widely =
  one or two subjects are driving the cluster structure; results are
  fragile.

**Alluvial / Sankey** (kinematic-cluster flow across phases): do
subjects stay in the same kinematic cluster across phases?

- Mostly horizontal flows (each subject stays in its column-aligned
  cluster across phases) = kinematic phenotype is stable across the
  experiment; subjects reach in their own characteristic style
  through baseline, injury, and recovery.
- Crossing flows = subjects change which other subjects they cluster
  with at different phases; kinematic structure reshuffles with
  injury/recovery.
- At small N the alluvial degenerates to one subject per cluster per
  phase; the visualization is a smoke test, not informative until
  N grows.

**Trajectory plots** (continuous + by-cluster):

- Subjects with similar connectivity PC1 score (similar color in the
  continuous view) following similar trajectories across phases =
  connectivity predicts kinematic trajectory shape.
- Similar-color subjects diverging across phases = no predictive
  relationship at small N.

**Interaction LMM table**: tests whether trajectory shape differs by
cluster. At small N with each cluster having ~1 subject the interaction is
vacuous; at higher N this becomes the primary inferential test for
"do connectivity clusters follow different recovery trajectories?".

</details>
